# 仮説検定の練習問題集 — t 検定から回帰分析・カイ二乗検定まで

このノートブックでは、**仮説検定**を例題 1 問 + 練習問題 11 問で練習します。
手計算(数式どおりの計算)で解いたうえで、`scipy.stats` や `statsmodels` で答え合わせをします。

| 問題 | 内容 | 検定の型 |
|---|---|---|
| 例題(解説付き) | ノートパソコンの平均購入価格 | 1 標本 t・片側 |
| 練習問題 1 | 学食の平均支払額 | 1 標本 t・片側 |
| 練習問題 2 | 2 つの学年の平均自習時間 | 2 標本 t・両側(Welch) |
| 練習問題 3 | 2 つの大学の動画配信サービス利用料金 | 2 標本 t・両側(Welch) |
| 練習問題 4 | 2 つの年代の電子書籍購入金額 | 2 標本 t・両側(Welch) |
| 練習問題 5 | 生成 AI サービス A・B の評価 | **対応のある t 検定**・両側 |
| 練習問題 6 | パッケージデザインの評価のばらつき | **分散の F 検定**・両側 |
| 練習問題 7 | 広告デザイン × 割引率の実験 | **二元配置分散分析**(反復あり) |
| 練習問題 8 | 広告費と新規顧客獲得数 | **単回帰分析**と傾きの t 検定 |
| 練習問題 9 | 生成 AI を毎週利用する学生の割合 | **母比率の z 検定**・両側 |
| 練習問題 10 | ランディングページの A/B テスト | **2 つの比率の差の z 検定**・両側 |
| 練習問題 11 | 学年と生成 AI 利用頻度の関連 | **カイ二乗独立性検定** |

## 例題

ある大学で、新入生のうちノートパソコンを新しく購入した **180 人**に、実際の購入価格について尋ねた。
その結果、平均購入価格は **142,500 円**、標本標準偏差は **18,000 円** であった。

> この大学の新入生が購入したノートパソコンの平均購入価格は、**140,000 円(比較値)より高い**といえるか。
> **有意水準 0.05** で検定しなさい。なお、**母標準偏差は不明**であるものとする。

### 解答する内容

1. 帰無仮説と対立仮説を設定する
2. 適切な検定方法を選ぶ
3. 検定統計量を計算する
4. 棄却域または p 値を求める
5. 帰無仮説を棄却できるか判断する
6. 分析結果を文章で説明する

このノートブックでは、手順を 1 つずつ確認しながら、計算は Python(`scipy.stats`)で行います。

**最初のセルは日本語フォントの読み込みのため、実行に数十秒かかります**。上から順に実行してください。

In [ ]:
import piplite
await piplite.install("matplotlib-fontja==1.1.0")

import matplotlib_fontja
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

matplotlib_fontja.japanize()

# 問題文で与えられた値
n = 180          # 標本の大きさ
x_bar = 142_500  # 標本平均(円)
s = 18_000       # 標本標準偏差(円)
mu0 = 140_000    # 比較値(円)
alpha = 0.05     # 有意水準

## 手順 1 — 帰無仮説と対立仮説を設定する

「140,000 円**より高い**といえるか」を確かめたいので、**片側検定(右側)** になります。

- **帰無仮説** $H_0: \mu = 140{,}000$ (平均購入価格は 140,000 円と変わらない)
- **対立仮説** $H_1: \mu > 140{,}000$ (平均購入価格は 140,000 円より高い)

検定では「$H_0$ が正しいとしたら、手元のデータはどれくらい珍しいか」を計算し、
十分珍しければ $H_0$ を棄却して $H_1$ を採択します。

## 手順 2 — 適切な検定方法を選ぶ

判断のポイントは 2 つです。

| 条件 | 今回 | 帰結 |
|---|---|---|
| 比べる対象 | 1 つの標本平均 と 基準値 140,000 円 | **1 標本の検定** |
| 母標準偏差 $\sigma$ | **不明**(標本標準偏差 $s$ しか無い) | z 検定ではなく **t 検定** |

したがって **1 標本 t 検定(片側・右側)** を使います。自由度は $n - 1 = 179$ です。

(標本が 180 人と大きいので t 分布は正規分布とほぼ同じ形になりますが、
「母標準偏差が不明なら t 検定」という原則どおりに進めます。)

## 手順 3 — 検定統計量を計算する

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$$

分母の $s/\sqrt{n}$ は**標準誤差**(標本平均のばらつきの大きさ)です。

In [ ]:
se = s / np.sqrt(n)          # 標準誤差
t_stat = (x_bar - mu0) / se  # 検定統計量
df = n - 1                   # 自由度

print(f"標準誤差 SE = {se:,.1f} 円")
print(f"検定統計量 t = {t_stat:.4f}")
print(f"自由度 df = {df}")

## 手順 4 — 棄却域と p 値を求める

**棄却域**: 右側の片側検定なので、自由度 179 の t 分布の上側 5% 点(臨界値)より
t が大きければ棄却します。

**p 値**: 「$H_0$ が正しいとき、今回の t 値以上の値が偶然出る確率」です。
これが有意水準 0.05 を下回れば棄却します。どちらの方法でも結論は同じになります。

In [ ]:
t_crit = stats.t.ppf(1 - alpha, df)  # 上側 5% 点(棄却域の境界)
p_value = stats.t.sf(t_stat, df)     # 上側確率 = p 値(片側)

print(f"棄却域: t > {t_crit:.4f}")
print(f"p 値 = {p_value:.4f}")

t 分布の上に、棄却域と今回の t 値を描いて位置関係を確かめます。

In [ ]:
x = np.linspace(-4, 4, 400)
y = stats.t.pdf(x, df)

plt.figure(figsize=(8, 3.5))
plt.plot(x, y, color="black")
plt.fill_between(x, y, where=(x >= t_crit), color="crimson", alpha=0.4,
                 label=f"棄却域 (t > {t_crit:.3f}, 面積 5%)")
plt.axvline(t_stat, color="seagreen", linewidth=2,
            label=f"今回の t = {t_stat:.3f}")
plt.title(f"自由度 {df} の t 分布と棄却域(片側検定)")
plt.xlabel("t")
plt.ylabel("確率密度")
plt.legend()
plt.tight_layout()
plt.show()

## 手順 5 — 帰無仮説を棄却できるか判断する

In [ ]:
print(f"t = {t_stat:.4f} は棄却域 (t > {t_crit:.4f}) に入っているか: {t_stat > t_crit}")
print(f"p 値 {p_value:.4f} は有意水準 {alpha} より小さいか: {p_value < alpha}")
print()
if p_value < alpha:
    print("→ 帰無仮説を棄却する(平均購入価格は 140,000 円より高いといえる)")
else:
    print("→ 帰無仮説を棄却できない(140,000 円より高いとはいえない)")

## 手順 6 — 分析結果を文章で説明する

> 新入生 180 人の標本(平均 142,500 円、標準偏差 18,000 円)について、
> 「平均購入価格は 140,000 円より高い」という仮説を 1 標本 t 検定(片側、有意水準 0.05)で検定した。
> 検定統計量は t(179) = 1.863、p 値は 0.032 であり、p 値が有意水準 0.05 を下回ったため帰無仮説は棄却された。
> したがって、**この大学の新入生のノートパソコンの平均購入価格は 140,000 円より統計的に有意に高いといえる**。

## 確認 — `scipy.stats` の関数で答え合わせ

要約統計量(平均・標準偏差)だけからの計算が正しいか、
同じ平均・標準偏差を持つデータを作って `stats.ttest_1samp()` に片側検定
(`alternative="greater"`)を指定し、結果が一致することを確かめます。

In [ ]:
# 平均 142,500・標本標準偏差 18,000 にぴったり一致する 180 個のデータを作る
rng = np.random.default_rng(0)
raw = rng.normal(0, 1, n)
data = (raw - raw.mean()) / raw.std(ddof=1) * s + x_bar

print(f"作成したデータ: 平均 {data.mean():,.1f} 円, 標本標準偏差 {data.std(ddof=1):,.1f} 円")

result = stats.ttest_1samp(data, popmean=mu0, alternative="greater")
print(f"scipy の結果: t = {result.statistic:.4f}, p 値 = {result.pvalue:.4f}")
print(f"手計算の結果: t = {t_stat:.4f}, p 値 = {p_value:.4f}")

t 値も p 値も一致しました。要約統計量しか与えられていない問題では今回のように手計算し、
生データがある場合は `stats.ttest_1samp()` を直接使えばよい、ということです。

## 補足 1 — 片側信頼区間で見る

「平均は少なくともいくら以上か」を区間で示すこともできます。
片側 95% 信頼区間の下限は $\bar{x} - t_{0.05,179} \times SE$ です。

In [ ]:
lower = x_bar - t_crit * se
print(f"母平均の片側 95% 信頼区間: {lower:,.0f} 円 以上")
print(f"比較値 140,000 円はこの区間の外(下)にある → 検定の結論と一致")

## 補足 2 — 「統計的に有意」と「差が大きい」は別の話

棄却されたのは「140,000 円と変わらない」という仮説であって、差の**大きさ**が保証されたわけではありません。
実際の差は 2,500 円で、平均購入価格の約 1.8% にすぎません。
標本が大きい(n = 180)ほど小さな差でも有意になりやすいため、
実務では p 値だけでなく**差の大きさ(効果量)** もあわせて報告するのが良い習慣です。

In [ ]:
cohens_d = (x_bar - mu0) / s
print(f"効果量 (Cohen の d) = {cohens_d:.3f}  → 一般的な目安では「小さい」差")

## 練習問題 1 — 自分で解いてみよう

同じ型の問題をもう 1 問。今度は自分の手で解いてみましょう。

> 「大学生が学食で昼食を購入した **36 人**の平均支払額 **1,640 円**(標準偏差 **420 円**)は、
> **1,500 円より高い**といえるか〈有意水準 5%〉」を、母平均の比較値との差の *t* 検定によって検証しなさい。
> なお、母標準偏差は不明であり、支払額は正規分布に従うものとする。

手順 1〜6 をそのままなぞれば解けます。
今回は標本が 36 人と小さいので、「支払額が正規分布に従う」という仮定が最初の問題より大事になります。

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 1)

- **手順 1**: $H_0: \mu = 1{,}500$ / $H_1: \mu > 1{,}500$(片側・右側)
- **手順 2**: 母標準偏差が不明で、支払額は正規分布に従う → **1 標本 t 検定**(自由度 $36-1=35$)

In [ ]:
n2 = 36
x_bar2 = 1_640
s2 = 420
mu0_2 = 1_500

se2 = s2 / np.sqrt(n2)             # 手順 3: 標準誤差
t_stat2 = (x_bar2 - mu0_2) / se2   #         検定統計量
df2 = n2 - 1

t_crit2 = stats.t.ppf(1 - alpha, df2)  # 手順 4: 棄却域の境界
p_value2 = stats.t.sf(t_stat2, df2)    #         p 値(片側)

print(f"手順 3: SE = {se2:.1f} 円,  t = {t_stat2:.4f},  df = {df2}")
print(f"手順 4: 棄却域 t > {t_crit2:.4f},  p 値 = {p_value2:.4f}")
print()
print(f"手順 5: t = {t_stat2:.2f} > {t_crit2:.4f} かつ p = {p_value2:.4f} < {alpha}")
if p_value2 < alpha:
    print("→ 帰無仮説を棄却する")
else:
    print("→ 帰無仮説を棄却できない")

- **手順 6(文章での説明)**:

> 学食で昼食を購入した 36 人の標本(平均 1,640 円、標準偏差 420 円)について、
> 「平均支払額は 1,500 円より高い」という仮説を 1 標本 t 検定(片側、有意水準 0.05)で検定した。
> 検定統計量は t(35) = 2.00、p 値は 0.027 であり、有意水準 0.05 を下回ったため帰無仮説は棄却された。
> したがって、**学生の平均支払額は 1,500 円より統計的に有意に高いといえる**。

計算のポイント: $\sqrt{36} = 6$ なので標準誤差はちょうど $420/6 = 70$ 円、
t 値は $140/70 = 2.00$ ときれいな値になります。
標本が小さいぶん臨界値は 1 問目(自由度 179 の 1.653)より大きい 1.690 になっている、
という t 分布の性質にも注目してください。

## 練習問題 2 — 2 つのグループの平均を比べる(Welch の t 検定)

最後は、1 つの平均と基準値ではなく、**2 つのグループの平均**を比べる問題です。

> 大学生の 1 週間当たりの自習時間について、**1 年生 80 人**と **3 年生 75 人**に尋ねた。
> その結果、1 年生では平均自習時間が **8.6 時間**、標準偏差が **3.2 時間**、
> 3 年生では平均自習時間が **10.1 時間**、標準偏差が **3.8 時間**であった。
> **両学年の平均自習時間に違いがあるといえるか。有意水準 0.05 で検定しなさい。**
> なお、2 つの標本は独立しており、母分散が等しいとは仮定せず、**Welch の 2 標本 t 検定**を用いるものとする。

例題との違いに注意しましょう。

- 「高いといえるか」ではなく「**違いがあるといえるか**」→ **両側検定**
- 標本が 2 つ → **2 標本 t 検定**。母分散が等しいと仮定しないので **Welch 流**を使う

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 2)

- **手順 1**: $H_0: \mu_1 = \mu_3$(両学年の平均は等しい)/ $H_1: \mu_1 \ne \mu_3$(**両側**)
- **手順 2**: 独立な 2 標本・母分散が等しい仮定なし → **Welch の 2 標本 t 検定**

検定統計量と自由度は次の式で計算します(自由度は Welch–Satterthwaite の近似式)。

$$t = \frac{\bar{x}_1 - \bar{x}_3}{\sqrt{s_1^2/n_1 + s_3^2/n_3}}, \qquad
\nu \approx \frac{(s_1^2/n_1 + s_3^2/n_3)^2}{\dfrac{(s_1^2/n_1)^2}{n_1 - 1} + \dfrac{(s_3^2/n_3)^2}{n_3 - 1}}$$

In [ ]:
n1, x1, s1 = 80, 8.6, 3.2    # 1 年生
n3, x3, s3 = 75, 10.1, 3.8   # 3 年生

v1, v3 = s1**2 / n1, s3**2 / n3
se_w = np.sqrt(v1 + v3)                                  # 手順 3: 標準誤差
t_w = (x1 - x3) / se_w                                   #         検定統計量
df_w = (v1 + v3) ** 2 / (v1**2 / (n1 - 1) + v3**2 / (n3 - 1))  # Welch の自由度

t_crit_w = stats.t.ppf(1 - alpha / 2, df_w)   # 手順 4: 両側なので上下 2.5% ずつ
p_w = 2 * stats.t.sf(abs(t_w), df_w)          #         p 値(両側)

print(f"手順 3: SE = {se_w:.4f} 時間,  t = {t_w:.4f},  自由度 ν ≈ {df_w:.1f}")
print(f"手順 4: 棄却域 |t| > {t_crit_w:.4f},  p 値(両側) = {p_w:.4f}")
print()
print(f"手順 5: |t| = {abs(t_w):.3f} > {t_crit_w:.4f} かつ p = {p_w:.4f} < {alpha}")
if p_w < alpha:
    print("→ 帰無仮説を棄却する(平均自習時間に違いがあるといえる)")
else:
    print("→ 帰無仮説を棄却できない")

scipy には要約統計量から直接 2 標本 t 検定を行う `ttest_ind_from_stats()` があるので、答え合わせをします。
`equal_var=False` が Welch 流の指定です。

In [ ]:
result_w = stats.ttest_ind_from_stats(
    mean1=x1, std1=s1, nobs1=n1,
    mean2=x3, std2=s3, nobs2=n3,
    equal_var=False,
)
print(f"scipy の結果: t = {result_w.statistic:.4f}, p 値 = {result_w.pvalue:.4f}")
print(f"手計算の結果: t = {t_w:.4f}, p 値 = {p_w:.4f}")

- **手順 6(文章での説明)**:

> 1 年生 80 人(平均 8.6 時間、標準偏差 3.2 時間)と 3 年生 75 人(平均 10.1 時間、標準偏差 3.8 時間)の
> 週間自習時間の差を、Welch の 2 標本 t 検定(両側、有意水準 0.05)で検定した。
> 検定統計量は t ≈ −2.65(自由度 ≈ 145)、p 値は 0.009 であり、有意水準 0.05 を下回ったため帰無仮説は棄却された。
> したがって、**両学年の平均自習時間には統計的に有意な違いがあり、3 年生の方が週あたり平均 1.5 時間長い**といえる。

なお、t が**負**になったのは「1 年生 − 3 年生」の順で差を取ったからで、
3 年生の方が長いことを表しています。両側検定なので符号はどちらでも構いません。

## 練習問題 3 — Welch の t 検定をもう 1 問

覚えたばかりの Welch の 2 標本 t 検定を、今度はヒントなしで解いてみましょう。

> 大学生の 1 か月当たりの動画配信サービス利用料金について、**A 大学の 45 人**と **B 大学の 38 人**に尋ねた。
> その結果、A 大学では平均利用料金が **2,480 円**、標準偏差が **620 円**、
> B 大学では平均利用料金が **2,120 円**、標準偏差が **540 円**であった。
> **両大学の学生の平均利用料金に差があるといえるか。有意水準 0.05 で検定しなさい。**
> なお、2 つの標本は独立しており、母分散が等しいとは仮定しないものとする。

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 3)

- **手順 1**: $H_0: \mu_A = \mu_B$ / $H_1: \mu_A \ne \mu_B$(「差があるといえるか」なので**両側**)
- **手順 2**: 独立 2 標本・等分散を仮定しない → **Welch の 2 標本 t 検定**

In [ ]:
nA, xA, sA = 45, 2_480, 620  # A 大学
nB, xB, sB = 38, 2_120, 540  # B 大学

vA, vB = sA**2 / nA, sB**2 / nB
se3 = np.sqrt(vA + vB)
t3 = (xA - xB) / se3
df3 = (vA + vB) ** 2 / (vA**2 / (nA - 1) + vB**2 / (nB - 1))
t_crit3 = stats.t.ppf(1 - alpha / 2, df3)
p3 = 2 * stats.t.sf(abs(t3), df3)

print(f"手順 3: SE = {se3:.2f} 円,  t = {t3:.4f},  自由度 ν ≈ {df3:.1f}")
print(f"手順 4: 棄却域 |t| > {t_crit3:.4f},  p 値(両側) = {p3:.4f}")

# scipy で答え合わせ
check = stats.ttest_ind_from_stats(mean1=xA, std1=sA, nobs1=nA,
                                   mean2=xB, std2=sB, nobs2=nB,
                                   equal_var=False)
print(f"scipy: t = {check.statistic:.4f}, p 値 = {check.pvalue:.4f}")
print()
print(f"手順 5: |t| = {abs(t3):.3f} > {t_crit3:.4f} かつ p = {p3:.4f} < {alpha}")
if p3 < alpha:
    print("→ 帰無仮説を棄却する(平均利用料金に差があるといえる)")
else:
    print("→ 帰無仮説を棄却できない")

- **手順 6(文章での説明)**:

> A 大学 45 人(平均 2,480 円、標準偏差 620 円)と B 大学 38 人(平均 2,120 円、標準偏差 540 円)の
> 動画配信サービス利用料金の差を、Welch の 2 標本 t 検定(両側、有意水準 0.05)で検定した。
> 検定統計量は t ≈ 2.83(自由度 ≈ 81)、p 値は 0.006 であり、有意水準 0.05 を下回ったため帰無仮説は棄却された。
> したがって、**両大学の平均利用料金には統計的に有意な差があり、A 大学の方が月あたり平均 360 円高い**といえる。

## 練習問題 4 — 総仕上げ

> 電子書籍の年間購入金額について、20〜59 歳の 80 人に尋ねた。
> その結果、**20〜39 歳の 32 人**では平均購入金額が **9,200 円**、標準偏差が **3,150 円**、
> **40〜59 歳の 48 人**では平均購入金額が **11,400 円**、標準偏差が **4,280 円**であった。
> **両年代の平均購入金額に差があるといえるか。有意水準 5% で検定しなさい。**
> なお、2 つの標本は独立しており、母分散が等しいとは仮定しないものとする。

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 4)

- **手順 1**: $H_0: \mu_{20\text{-}39} = \mu_{40\text{-}59}$ / $H_1: \mu_{20\text{-}39} \ne \mu_{40\text{-}59}$(両側)
- **手順 2**: 独立 2 標本・等分散を仮定しない → **Welch の 2 標本 t 検定**

In [ ]:
nY, xY, sY = 32, 9_200, 3_150    # 20〜39 歳
nO, xO, sO = 48, 11_400, 4_280   # 40〜59 歳

vY, vO = sY**2 / nY, sO**2 / nO
se4 = np.sqrt(vY + vO)
t4 = (xY - xO) / se4
df4 = (vY + vO) ** 2 / (vY**2 / (nY - 1) + vO**2 / (nO - 1))
t_crit4 = stats.t.ppf(1 - alpha / 2, df4)
p4 = 2 * stats.t.sf(abs(t4), df4)

print(f"手順 3: SE = {se4:.2f} 円,  t = {t4:.4f},  自由度 ν ≈ {df4:.1f}")
print(f"手順 4: 棄却域 |t| > {t_crit4:.4f},  p 値(両側) = {p4:.4f}")

check4 = stats.ttest_ind_from_stats(mean1=xY, std1=sY, nobs1=nY,
                                    mean2=xO, std2=sO, nobs2=nO,
                                    equal_var=False)
print(f"scipy: t = {check4.statistic:.4f}, p 値 = {check4.pvalue:.4f}")
print()
print(f"手順 5: |t| = {abs(t4):.3f} > {t_crit4:.4f} かつ p = {p4:.4f} < {alpha}")
if p4 < alpha:
    print("→ 帰無仮説を棄却する(平均購入金額に差があるといえる)")
else:
    print("→ 帰無仮説を棄却できない")

- **手順 6(文章での説明)**:

> 20〜39 歳 32 人(平均 9,200 円、標準偏差 3,150 円)と 40〜59 歳 48 人(平均 11,400 円、標準偏差 4,280 円)の
> 電子書籍の年間購入金額の差を、Welch の 2 標本 t 検定(両側、有意水準 0.05)で検定した。
> 検定統計量は t ≈ −2.65(自由度 ≈ 77)、p 値は 0.010 であり、有意水準 0.05 を下回ったため帰無仮説は棄却された。
> したがって、**両年代の平均購入金額には統計的に有意な差があり、40〜59 歳の方が年間平均 2,200 円多い**といえる。

標本の大きさや標準偏差が違っても、Welch の式に当てはめる流れはまったく同じです。

## 練習問題 5 — 対応のある t 検定

最後は、**同じ人が 2 つの条件を評価した**データです。これまでの「独立な 2 標本」とは扱いが変わります。

> 150 人の大学生に、2 種類の生成 AI サービス A と B の使いやすさを、それぞれ 10 点満点で評価してもらったところ、
> 次のデータを得た。**A と B の平均評価に差があるかどうかを、有意水準 0.05 で検定しなさい。**
> なお、同じ学生が両方のサービスを評価しているため、**対応のある標本の t 検定**を用いること。

| 学生 $n$ | $A$ の評価 $x_i$ | $B$ の評価 $y_i$ | $d_i = x_i - y_i$ | $d_i^2$ |
|---:|---:|---:|---:|---:|
| 1 | 8 | 7 | $+1$ | 1 |
| 2 | 6 | 8 | $-2$ | 4 |
| 3 | 9 | 6 | $+3$ | 9 |
| $\vdots$ | $\vdots$ | $\vdots$ | $\vdots$ | $\vdots$ |
| 150 | 7 | 7 | $0$ | 0 |
| **合計** | | | **120** | **720** |

**考え方**: 同じ学生のペアなので、学生ごとの**差 $d_i = x_i - y_i$** を取ってしまえば、
「差の平均は 0 といえるか」という **1 標本 t 検定**に帰着します。
表には $\sum d_i = 120$ と $\sum d_i^2 = 720$ が与えられているので、そこから差の平均と標準偏差を計算できます。

$$\bar{d} = \frac{\sum d_i}{n}, \qquad
s_d^2 = \frac{\sum d_i^2 - n\bar{d}^2}{n - 1}, \qquad
t = \frac{\bar{d} - 0}{s_d / \sqrt{n}}$$

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 5)

- **手順 1**: $H_0: \mu_d = 0$(差の平均は 0)/ $H_1: \mu_d \ne 0$(両側)
- **手順 2**: 同じ学生による対応のあるデータ → 差を取って **1 標本 t 検定**(自由度 $150-1=149$)

In [ ]:
n5 = 150
sum_d = 120
sum_d2 = 720

d_bar = sum_d / n5                                # 差の平均
s_d = np.sqrt((sum_d2 - n5 * d_bar**2) / (n5 - 1))  # 差の標本標準偏差
se5 = s_d / np.sqrt(n5)
t5 = (d_bar - 0) / se5
df5 = n5 - 1
t_crit5 = stats.t.ppf(1 - alpha / 2, df5)
p5 = 2 * stats.t.sf(abs(t5), df5)

print(f"差の平均 d̄ = {d_bar:.3f} 点,  差の標準偏差 s_d = {s_d:.4f} 点")
print(f"手順 3: SE = {se5:.4f},  t = {t5:.4f},  df = {df5}")
print(f"手順 4: 棄却域 |t| > {t_crit5:.4f},  p 値(両側) = {p5:.6f}")
print()
print(f"手順 5: |t| = {abs(t5):.2f} > {t_crit5:.4f} かつ p < {alpha}")
if p5 < alpha:
    print("→ 帰無仮説を棄却する(平均評価に差があるといえる)")
else:
    print("→ 帰無仮説を棄却できない")

`scipy` で答え合わせをします。差の値だけが分かっている場合、
対応のある t 検定は「差に対する `ttest_1samp(d, popmean=0)`」とまったく同じです
(生データのペアがあるときは `stats.ttest_rel(x, y)` を使います)。

In [ ]:
# 平均 0.8・標本標準偏差 s_d にぴったり一致する差データを 150 個作って確認
rng5 = np.random.default_rng(1)
raw5 = rng5.normal(0, 1, n5)
d = (raw5 - raw5.mean()) / raw5.std(ddof=1) * s_d + d_bar

result5 = stats.ttest_1samp(d, popmean=0)
print(f"scipy の結果: t = {result5.statistic:.4f}, p 値 = {result5.pvalue:.6f}")
print(f"手計算の結果: t = {t5:.4f}, p 値 = {p5:.6f}")

- **手順 6(文章での説明)**:

> 150 人の大学生による生成 AI サービス A・B の評価の差を、対応のある t 検定(両側、有意水準 0.05)で検定した。
> 差の平均は 0.80 点、検定統計量は t(149) = 4.79、p 値は 0.001 未満であり、帰無仮説は棄却された。
> したがって、**A と B の平均評価には統計的に有意な差があり、A の方が平均 0.8 点高い**といえる。

**独立 2 標本との違いに注意**: もし対応を無視して Welch の検定をしてしまうと、
「同じ学生の甘口・辛口」という個人差のばらつきが混ざり、検定の感度が落ちます。
同じ対象を 2 回測ったデータでは、必ず対応のある検定を使いましょう。

## 練習問題 6 — 分散の差の検定(F 検定)

最後は、平均ではなく**ばらつき(分散)** を比べる問題です。マーケティング・リサーチの設定で出題します。

> ある飲料メーカーは、新商品のパッケージデザイン案 A と B に対する消費者評価の**ばらつき**を比較するため、
> マーケティング調査を実施した。消費者を無作為に 2 つのグループに分け、それぞれ **50 人**に異なるデザインを提示し、
> 購入意向を 7 段階で評価してもらった。その結果、購入意向得点の**標本分散**は、
> デザイン A が **2.250**、デザイン B が **1.125** であった。
> **デザイン A と B では、購入意向得点の分散に差があるといえるか。有意水準 0.05 で検定しなさい。**
> なお、2 つの標本は独立しており、購入意向得点はそれぞれ正規分布に従うものとする。

**考え方**: 正規分布に従う独立な 2 標本の分散比

$$F = \frac{s_A^2}{s_B^2}$$

は、帰無仮説(分散が等しい)のもとで**自由度 $(n_A - 1,\ n_B - 1)$ の F 分布**に従います。
両側検定なので、F 分布の上側 2.5% 点より大きいか、下側 2.5% 点より小さければ棄却します。

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 6)

- **手順 1**: $H_0: \sigma_A^2 = \sigma_B^2$ / $H_1: \sigma_A^2 \ne \sigma_B^2$(両側)
- **手順 2**: 独立 2 標本・それぞれ正規分布 → **分散比の F 検定**(自由度 $(49, 49)$)

In [ ]:
nA6, varA = 50, 2.250   # デザイン A
nB6, varB = 50, 1.125   # デザイン B

F = varA / varB
dfA, dfB = nA6 - 1, nB6 - 1

f_upper = stats.f.ppf(1 - alpha / 2, dfA, dfB)  # 上側 2.5% 点
f_lower = stats.f.ppf(alpha / 2, dfA, dfB)      # 下側 2.5% 点
p6 = 2 * min(stats.f.sf(F, dfA, dfB), stats.f.cdf(F, dfA, dfB))  # 両側 p 値

print(f"手順 3: F = {F:.4f}  (自由度 ({dfA}, {dfB}))")
print(f"手順 4: 棄却域 F < {f_lower:.4f} または F > {f_upper:.4f},  p 値(両側) = {p6:.4f}")
print()
print(f"手順 5: F = {F:.2f} > {f_upper:.4f} かつ p = {p6:.4f} < {alpha}")
if p6 < alpha:
    print("→ 帰無仮説を棄却する(分散に差があるといえる)")
else:
    print("→ 帰無仮説を棄却できない")

- **手順 6(文章での説明)**:

> デザイン A(50 人、標本分散 2.250)とデザイン B(50 人、標本分散 1.125)の購入意向得点のばらつきの差を、
> 分散比の F 検定(両側、有意水準 0.05)で検定した。検定統計量は F(49, 49) = 2.00、p 値は 0.017 であり、
> 有意水準 0.05 を下回ったため帰無仮説は棄却された。したがって、
> **両デザインの評価の分散には統計的に有意な差があり、デザイン A の方が消費者の評価が分かれている**といえる。

マーケティングの解釈としては、「A は好みが割れるとがったデザイン、B は無難で評価が安定したデザイン」
という読み方ができます。平均だけでなくばらつきを比べると、こうした違いが見えてきます。

**注意**: この F 検定は**正規分布の仮定に敏感**です。生データがある場合は、
正規性からのずれに強い Levene 検定(`stats.levene`)などがよく使われます。
また、練習問題 2〜4 で最初から Welch の t 検定を使ったように、
「まず F 検定で等分散を確かめてから t 検定を選ぶ」二段構えは現在は推奨されていません。

## 練習問題 7 — 二元配置分散分析(反復あり)

総仕上げは、**2 つの要因**の効果を同時に調べる**分散分析(ANOVA)** です。

> ある企業が、3 種類の Web 広告デザイン $A_1, A_2, A_3$ と、3 水準の割引率 $B_1, B_2, B_3$
> ($B_1$: 割引なし、$B_2$: 10% 割引、$B_3$: 20% 割引)を組み合わせた実験を行った。
> 各組み合わせについて 4 回ずつ広告を配信したところ、購入件数は次のようになった。
> **反復のある二元配置分散分析を行い、有意水準 0.05 で次の点を検定しなさい。**
>
> 1. 広告デザインによって平均購入件数に違いがあるか
> 2. 割引率によって平均購入件数に違いがあるか
> 3. 広告デザインと割引率の間に交互作用があるか

| 反復 | $A_1B_1$ | $A_1B_2$ | $A_1B_3$ | $A_2B_1$ | $A_2B_2$ | $A_2B_3$ | $A_3B_1$ | $A_3B_2$ | $A_3B_3$ |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 42 | 50 | 58 | 47 | 57 | 67 | 38 | 46 | 52 |
| 2 | 45 | 52 | 61 | 49 | 55 | 64 | 41 | 44 | 55 |
| 3 | 39 | 48 | 57 | 46 | 59 | 66 | 39 | 48 | 53 |
| 4 | 46 | 54 | 60 | 50 | 57 | 67 | 42 | 46 | 56 |
| **平均** | **43** | **51** | **59** | **48** | **57** | **66** | **40** | **46** | **54** |

**考え方**: データ全体のばらつき(平方和)を
「デザインの効果 $SS_A$」「割引率の効果 $SS_B$」「交互作用 $SS_{AB}$」「誤差 $SS_E$」に分解し、
それぞれを誤差と比べる F 検定を 3 回行います。
**交互作用**とは「割引の効き方がデザインによって変わる」ような組み合わせ効果のことです。

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 7)

- **手順 1**(仮説は 3 組):
  1. $H_0$: デザインの母平均はすべて等しい / $H_1$: どこかに違いがある
  2. $H_0$: 割引率の母平均はすべて等しい / $H_1$: どこかに違いがある
  3. $H_0$: 交互作用はない / $H_1$: 交互作用がある
- **手順 2**: 2 要因 × 各セル 4 反復の実験データ → **反復のある二元配置分散分析**

まずデータを縦長の DataFrame にして、セルごとの平均を確認します。

In [ ]:
import pandas as pd

values = {
    ("A1", "B1"): [42, 45, 39, 46], ("A1", "B2"): [50, 52, 48, 54], ("A1", "B3"): [58, 61, 57, 60],
    ("A2", "B1"): [47, 49, 46, 50], ("A2", "B2"): [57, 55, 59, 57], ("A2", "B3"): [67, 64, 66, 67],
    ("A3", "B1"): [38, 41, 39, 42], ("A3", "B2"): [46, 44, 48, 46], ("A3", "B3"): [52, 55, 53, 56],
}
ad = pd.DataFrame(
    [(a, b, v) for (a, b), vs in values.items() for v in vs],
    columns=["デザイン", "割引率", "購入件数"],
)
print(f"{len(ad)} 件のデータ(3 デザイン × 3 割引率 × 4 反復)")
ad.pivot_table(index="デザイン", columns="割引率", values="購入件数", aggfunc="mean")

**交互作用プロット**(割引率ごとの平均を、デザイン別の折れ線で描いた図)を見ると、
3 本の線がほぼ平行かどうかで、交互作用の有無を目で確かめられます。

In [ ]:
cell_means = ad.pivot_table(index="割引率", columns="デザイン", values="購入件数", aggfunc="mean")

plt.figure(figsize=(6.5, 3.5))
for design in cell_means.columns:
    plt.plot(cell_means.index, cell_means[design], marker="o", label=design)
plt.title("交互作用プロット — 線がほぼ平行なら交互作用は小さい")
plt.xlabel("割引率")
plt.ylabel("平均購入件数")
plt.legend(title="デザイン")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**手順 3〜4**: 平方和を分解して F 値と p 値を計算します(式のとおりの手計算)。

In [ ]:
y = ad["購入件数"].to_numpy(dtype=float)
grand = y.mean()

a_means = ad.groupby("デザイン")["購入件数"].mean()
b_means = ad.groupby("割引率")["購入件数"].mean()
ab_means = ad.groupby(["デザイン", "割引率"])["購入件数"].mean()

n_rep, n_a, n_b = 4, 3, 3
ss_a = n_rep * n_b * ((a_means - grand) ** 2).sum()
ss_b = n_rep * n_a * ((b_means - grand) ** 2).sum()
ss_ab = n_rep * sum(
    (ab_means[a, b] - a_means[a] - b_means[b] + grand) ** 2
    for a in a_means.index for b in b_means.index
)
ss_e = sum(
    ((ad.query("デザイン == @a and 割引率 == @b")["購入件数"] - ab_means[a, b]) ** 2).sum()
    for a in a_means.index for b in b_means.index
)

df_a, df_b = n_a - 1, n_b - 1
df_ab = df_a * df_b
df_e = n_a * n_b * (n_rep - 1)
ms_a, ms_b, ms_ab, ms_e = ss_a / df_a, ss_b / df_b, ss_ab / df_ab, ss_e / df_e

for name, ss, dfx, ms in [("デザイン (A)", ss_a, df_a, ms_a), ("割引率 (B)", ss_b, df_b, ms_b),
                          ("交互作用 (A×B)", ss_ab, df_ab, ms_ab), ("誤差", ss_e, df_e, ms_e)]:
    line = f"{name:<12} SS = {ss:8.2f}  df = {dfx:2d}  MS = {ms:7.2f}"
    if name != "誤差":
        f_val = ms / ms_e
        p_val = stats.f.sf(f_val, dfx, df_e)
        line += f"  F = {f_val:7.2f}  p = {p_val:.4f}"
    print(line)

`statsmodels` の分散分析で答え合わせをします(釣り合いの取れたデータなので結果は一致します)。

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols("購入件数 ~ C(デザイン) * C(割引率)", data=ad).fit()
sm.stats.anova_lm(model, typ=2).round(4)

**手順 5〜6(判断と文章での説明)**:

> 反復のある二元配置分散分析(有意水準 0.05)の結果、
> **広告デザインの主効果**は F(2, 27) ≈ 77.9(p < 0.001)、
> **割引率の主効果**は F(2, 27) ≈ 185.3(p < 0.001)でいずれも有意であり、帰無仮説は棄却された。
> 一方、**交互作用**は F(4, 27) ≈ 1.07(p ≈ 0.39)で有意ではなく、交互作用があるとはいえない。
> したがって、**(1) デザインによって、(2) 割引率によって平均購入件数は変わるが、
> (3) 割引の効き方がデザインによって変わるという証拠はない**、と結論できる。

マーケティングの解釈: デザインは $A_2$ が最も購入件数が多く、割引率は高いほど購入が増えます。
交互作用が無いので「どのデザインでも割引の上乗せ効果はほぼ同じ」→ デザイン選びと割引率は
**別々に最適化してよい**、という実務上シンプルな結論になります(交互作用プロットの平行な線がその表れです)。

## 練習問題 8 — 単回帰分析

最後は、**数値どうしの関係**を式で表す**単回帰分析**です。

> あるオンラインショップが 16 地域で実施したデジタル広告について、
> 広告費 $x$(万円)と新規顧客獲得数 $y$(人)を調査した(データは下の表)。
> 単回帰分析を行い、次の問いに答えなさい。
>
> 1. 広告費を説明変数、新規顧客獲得数を目的変数とする**散布図**を作成しなさい
> 2. 広告費と新規顧客獲得数の**相関係数**を求めなさい
> 3. 単回帰式 $\hat{y} = b_0 + b_1 x$ を推定しなさい
> 4. 回帰係数 $b_1$ がゼロではないといえるか、$H_0: \beta_1 = 0$ / $H_1: \beta_1 \ne 0$ を有意水準 0.05 で検定しなさい
> 5. **決定係数** $R^2$ を求め、その意味を説明しなさい
> 6. 広告費が **18 万円**の地域における新規顧客獲得数を予測しなさい
> 7. 回帰係数の意味を、マーケティング担当者に説明する文章としてまとめなさい
> 8. この結果だけから「広告費を増やせば新規顧客が増える」という**因果関係**を断定できるか

| 地域 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 広告費 $x$ | 6 | 9 | 12 | 8 | 15 | 18 | 14 | 20 | 11 | 17 | 22 | 13 | 19 | 24 | 10 | 16 |
| 顧客数 $y$ | 82 | 95 | 111 | 89 | 130 | 151 | 125 | 162 | 103 | 143 | 176 | 118 | 156 | 191 | 98 | 136 |

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 8)

**問 1(散布図)** — まずデータを眺めます。直線的な関係が見えるはずです。

In [ ]:
x8 = np.array([6, 9, 12, 8, 15, 18, 14, 20, 11, 17, 22, 13, 19, 24, 10, 16], dtype=float)
y8 = np.array([82, 95, 111, 89, 130, 151, 125, 162, 103, 143, 176, 118, 156, 191, 98, 136], dtype=float)

plt.figure(figsize=(6, 4))
plt.scatter(x8, y8, color="steelblue")
plt.title("広告費と新規顧客獲得数の散布図(16 地域)")
plt.xlabel("広告費(万円)")
plt.ylabel("新規顧客獲得数(人)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**問 2(相関係数)と問 3(回帰式)** — 偏差の積和 $S_{xy}$ と平方和 $S_{xx}$ から計算します。

$$b_1 = \frac{S_{xy}}{S_{xx}}, \qquad b_0 = \bar{y} - b_1 \bar{x}$$

In [ ]:
n8 = len(x8)
sxx = ((x8 - x8.mean()) ** 2).sum()
sxy = ((x8 - x8.mean()) * (y8 - y8.mean())).sum()

r8 = np.corrcoef(x8, y8)[0, 1]        # 問 2: 相関係数
b1 = sxy / sxx                        # 問 3: 傾き
b0 = y8.mean() - b1 * x8.mean()       #        切片

print(f"問 2: 相関係数 r = {r8:.4f}")
print(f"問 3: 回帰式  ŷ = {b0:.2f} + {b1:.3f} x")

**問 4(傾きの t 検定)** — 残差から誤差分散を推定し、$t = b_1 / SE(b_1)$ を自由度 $n-2$ の t 分布で検定します。

In [ ]:
y_hat8 = b0 + b1 * x8
sse8 = ((y8 - y_hat8) ** 2).sum()
mse8 = sse8 / (n8 - 2)
se_b1 = np.sqrt(mse8 / sxx)

t8 = b1 / se_b1
df8 = n8 - 2
t_crit8 = stats.t.ppf(1 - alpha / 2, df8)
p8 = 2 * stats.t.sf(abs(t8), df8)

print(f"SE(b1) = {se_b1:.4f},  t = {t8:.2f},  df = {df8}")
print(f"棄却域 |t| > {t_crit8:.4f},  p 値 = {p8:.2e}")

# scipy で答え合わせ
res8 = stats.linregress(x8, y8)
print(f"scipy: 傾き = {res8.slope:.3f}, 切片 = {res8.intercept:.2f}, r = {res8.rvalue:.4f}, p 値 = {res8.pvalue:.2e}")
print()
if p8 < alpha:
    print("→ 帰無仮説を棄却する(傾きはゼロではない)")
else:
    print("→ 帰無仮説を棄却できない")

**問 5(決定係数)と問 6(予測)**

In [ ]:
r2 = 1 - sse8 / ((y8 - y8.mean()) ** 2).sum()
pred_18 = b0 + b1 * 18

print(f"問 5: 決定係数 R² = {r2:.4f}")
print(f"問 6: 広告費 18 万円の地域の予測顧客数 = {pred_18:.1f} 人")

plt.figure(figsize=(6, 4))
plt.scatter(x8, y8, color="steelblue", label="観測データ")
xs = np.linspace(5, 25, 100)
plt.plot(xs, b0 + b1 * xs, color="crimson", label=f"回帰直線 y = {b0:.1f} + {b1:.2f}x")
plt.scatter([18], [pred_18], color="darkorange", s=120, zorder=3, marker="*",
            label=f"x=18 の予測 ({pred_18:.0f} 人)")
plt.title("回帰直線と予測")
plt.xlabel("広告費(万円)")
plt.ylabel("新規顧客獲得数(人)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**問 5 の意味**: $R^2 = 0.994$ は「新規顧客獲得数のばらつきの約 **99.4%** が、
広告費の違いによって説明できる」ことを意味します(1 に近いほど直線の当てはまりが良い)。

**問 7(マーケティング担当者への説明文)**:

> 16 地域のデータを分析したところ、広告費と新規顧客獲得数の間には非常に強い正の関係がありました
> (相関係数 0.997)。回帰式は「予測顧客数 = 38.3 + 6.21 × 広告費(万円)」で、
> **広告費を 1 万円増やすごとに、新規顧客が平均で約 6.2 人増える**関係です(統計的に有意、p < 0.001)。
> なお切片の 38.3 人は「広告費ゼロなら 38 人獲得できる」という意味に読みたくなりますが、
> 今回のデータには広告費 6 万円未満の地域が無いため、**範囲外への当てはめ(外挿)は避けるべき**です。

**問 8(因果関係を断定できるか)** — **断定できません。** 理由は次のとおりです。

- これは実験ではなく**観察データ**です。地域は無作為に広告費を割り当てられていません
- **交絡要因**の可能性: 例えば**地域人口や市場規模**が大きい地域ほど、広告予算も多く配分され、
  かつ(広告と無関係に)新規顧客も多く獲得できるとすれば、両者の相関は人口の影響で生じている可能性があります
- **競合状況**や景気、店舗網なども同様に、広告費と顧客数の両方に影響し得ます
- 因果を確かめるには、練習問題 7 のような**無作為化した実験**(地域をランダムに分けて広告費を変える
  A/B テスト)や、人口・市場規模を説明変数に加えた**重回帰分析**などが必要です

「相関は因果を意味しない」— 回帰分析の結果を報告するときに、必ず添えるべき注意です。

## 練習問題 9 — 母比率の検定

ここからは、平均ではなく**割合(比率)** を扱います。「はい / いいえ」で答えるタイプのデータの検定です。

> 全国調査によると、大学生のうち生成 AI を**毎週利用する**人の割合は **30%** だという。
> ある大学で無作為に選んだ **400 人**に尋ねたところ、**148 人**が毎週利用と回答した。
> **この大学の割合は全国の 30% と異なるといえるか。有意水準 0.05 で検定しなさい。**

**考え方**: 標本比率 $\hat{p} = x/n$ は、$n$ が大きければ正規分布で近似できます
(目安: $np_0 \ge 10$ かつ $n(1-p_0) \ge 10$)。帰無仮説のもとでの標準誤差を使い、

$$z = \frac{\hat{p} - p_0}{\sqrt{p_0(1-p_0)/n}}$$

を標準正規分布で検定します(比率の検定では t ではなく **z 検定**を使うのが標準です)。

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 9)

- **手順 1**: $H_0: p = 0.30$ / $H_1: p \ne 0.30$(両側)
- **手順 2**: 大標本の比率($np_0 = 120 \ge 10$、$n(1-p_0) = 280 \ge 10$)→ **母比率の z 検定**

In [ ]:
n9, x9, p0 = 400, 148, 0.30
p_hat = x9 / n9

z9 = (p_hat - p0) / np.sqrt(p0 * (1 - p0) / n9)
z_crit9 = stats.norm.ppf(1 - alpha / 2)
p9 = 2 * stats.norm.sf(abs(z9))

print(f"標本比率 p̂ = {p_hat:.3f} ({x9}/{n9} 人)")
print(f"手順 3: z = {z9:.4f}")
print(f"手順 4: 棄却域 |z| > {z_crit9:.4f},  p 値(両側) = {p9:.4f}")

# 正規近似を使わない「厳密な」二項検定でも答え合わせ
exact9 = stats.binomtest(x9, n9, p0)
print(f"scipy(二項検定・厳密): p 値 = {exact9.pvalue:.4f}")
print()
print(f"手順 5: |z| = {abs(z9):.3f} > {z_crit9:.4f} かつ p = {p9:.4f} < {alpha}")
if p9 < alpha:
    print("→ 帰無仮説を棄却する(全国の 30% と異なるといえる)")
else:
    print("→ 帰無仮説を棄却できない")

- **手順 6(文章での説明)**:

> 無作為に選んだ 400 人のうち 148 人(37.0%)が毎週利用と回答した。
> 母比率の z 検定(両側、有意水準 0.05)の結果、z = 3.06、p 値は 0.002 であり、帰無仮説は棄却された。
> したがって、**この大学の毎週利用率は全国の 30% と統計的に有意に異なり、7 ポイント高い**といえる。

正規近似の z 検定と、近似を使わない二項検定(`stats.binomtest`)の p 値がほぼ一致していることも確認できました。
標本が小さいときは二項検定を使うのが安全です。

## 練習問題 10 — 2 つの比率の差(A/B テスト)

マーケティングで最も出番が多い検定です。

> あるオンラインショップが、ランディングページの新デザインを試す **A/B テスト**を行った。
> 訪問者を無作為に 2 群に分けたところ、**A 案は 1,200 人中 96 人**(8.0%)が、
> **B 案は 1,000 人中 55 人**(5.5%)が商品購入ボタンをクリックした。
> **A 案と B 案でクリック率に差があるといえるか。有意水準 0.05 で検定しなさい。**

**考え方**: 帰無仮説「$p_A = p_B$」のもとでは共通の比率が想定できるので、
2 群を**まとめた比率(プール比率)** $\bar{p}$ で標準誤差を作ります。

$$\bar{p} = \frac{x_A + x_B}{n_A + n_B}, \qquad
z = \frac{\hat{p}_A - \hat{p}_B}{\sqrt{\bar{p}(1-\bar{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 10)

- **手順 1**: $H_0: p_A = p_B$ / $H_1: p_A \ne p_B$(両側)
- **手順 2**: 独立な 2 群の大標本比率 → **2 つの比率の差の z 検定**(プール比率を使用)

In [ ]:
nA10, xA10 = 1_200, 96   # A 案
nB10, xB10 = 1_000, 55   # B 案

pA = xA10 / nA10
pB = xB10 / nB10
p_pool = (xA10 + xB10) / (nA10 + nB10)

se10 = np.sqrt(p_pool * (1 - p_pool) * (1 / nA10 + 1 / nB10))
z10 = (pA - pB) / se10
p10 = 2 * stats.norm.sf(abs(z10))

print(f"クリック率: A 案 {pA:.1%}, B 案 {pB:.1%}, プール比率 {p_pool:.4f}")
print(f"手順 3: z = {z10:.4f}")
print(f"手順 4: p 値(両側) = {p10:.4f}")

# 答え合わせ: 2×2 分割表のカイ二乗検定(補正なし)は z 検定と完全に一致する(χ² = z²)
table10 = [[xA10, nA10 - xA10], [xB10, nB10 - xB10]]
chi2_10 = stats.chi2_contingency(table10, correction=False)
print(f"scipy(カイ二乗): χ² = {chi2_10.statistic:.4f} (= z² = {z10**2:.4f}), p 値 = {chi2_10.pvalue:.4f}")
print()
if p10 < alpha:
    print(f"手順 5: p = {p10:.4f} < {alpha} → 帰無仮説を棄却する(クリック率に差があるといえる)")
else:
    print(f"手順 5: p = {p10:.4f} ≥ {alpha} → 帰無仮説を棄却できない")

- **手順 6(文章での説明)**:

> A/B テストの結果(A 案 8.0%、B 案 5.5%)について、2 つの比率の差の z 検定(両側、有意水準 0.05)を行った。
> z = 2.31、p 値は 0.021 であり、帰無仮説は棄却された。したがって、
> **A 案のクリック率は B 案より統計的に有意に高く(+2.5 ポイント)、A 案を採用する根拠になる**といえる。

**豆知識**: 2×2 分割表のカイ二乗検定(連続性補正なし)の統計量は、この z 検定の 2 乗($\chi^2 = z^2$)に
ぴったり一致します。答え合わせで確認したとおりです。
また、A/B テストは訪問者を**無作為に割り当てた実験**なので、練習問題 8 の観察データと違い、
差が出れば**因果的な効果**として解釈できるのが強みです。

## 練習問題 11 — カイ二乗独立性検定

最後は、**カテゴリ × カテゴリ**の関連を調べる検定です。クロス集計表(分割表)がそのまま検定の材料になります
(クロス集計表の作り方は `tutorials/pandas-crosstab.ipynb` で練習できます)。

> 大学生 300 人に、学年(1・2 年生 / 3・4 年生)と生成 AI の利用頻度(未経験 / 時々 / 毎週)を尋ねたところ、
> 次の分割表を得た。**学年と利用頻度の間に関連があるといえるか。有意水準 0.05 で検定しなさい。**

| | 未経験 | 時々 | 毎週 | 合計 |
|---|---:|---:|---:|---:|
| 1・2 年生 | 40 | 70 | 50 | 160 |
| 3・4 年生 | 20 | 50 | 70 | 140 |
| **合計** | **60** | **120** | **120** | **300** |

**考え方**: 「関連がない(独立)」と仮定したときの**期待度数**
$E_{ij} = \dfrac{\text{行合計} \times \text{列合計}}{\text{総数}}$ を計算し、観測度数とのずれを

$$\chi^2 = \sum \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

に集めて、自由度 $(行数-1)(列数-1)$ のカイ二乗分布で検定します。

In [ ]:
# ここにコードを書いてみましょう


### 解答例(練習問題 11)

- **手順 1**: $H_0$: 学年と利用頻度は独立(関連がない)/ $H_1$: 独立ではない(関連がある)
- **手順 2**: 2×3 の分割表 → **カイ二乗独立性検定**(自由度 $(2-1)(3-1) = 2$)

In [ ]:
observed = np.array([
    [40, 70, 50],   # 1・2 年生
    [20, 50, 70],   # 3・4 年生
])

row_totals = observed.sum(axis=1, keepdims=True)
col_totals = observed.sum(axis=0, keepdims=True)
total = observed.sum()
expected = row_totals @ col_totals / total

print("期待度数(独立と仮定した場合):")
print(pd.DataFrame(expected, index=["1・2 年生", "3・4 年生"], columns=["未経験", "時々", "毎週"]))

chi2_11 = ((observed - expected) ** 2 / expected).sum()
df11 = (observed.shape[0] - 1) * (observed.shape[1] - 1)
chi2_crit = stats.chi2.ppf(1 - alpha, df11)
p11 = stats.chi2.sf(chi2_11, df11)

print()
print(f"手順 3: χ² = {chi2_11:.4f}  (自由度 {df11}、期待度数はすべて 5 以上 → 近似 OK)")
print(f"手順 4: 棄却域 χ² > {chi2_crit:.4f},  p 値 = {p11:.4f}")

# scipy で答え合わせ
res11 = stats.chi2_contingency(observed)
print(f"scipy: χ² = {res11.statistic:.4f}, p 値 = {res11.pvalue:.4f}")
print()
print(f"手順 5: χ² = {chi2_11:.2f} > {chi2_crit:.4f} かつ p = {p11:.4f} < {alpha}")
if p11 < alpha:
    print("→ 帰無仮説を棄却する(学年と利用頻度に関連があるといえる)")
else:
    print("→ 帰無仮説を棄却できない")

- **手順 6(文章での説明)**:

> 大学生 300 人の分割表についてカイ二乗独立性検定(有意水準 0.05)を行ったところ、
> χ²(2) = 12.05、p 値は 0.002 であり、帰無仮説は棄却された。
> したがって、**学年と生成 AI の利用頻度には統計的に有意な関連がある**といえる。
> 観測度数と期待度数を比べると、3・4 年生では「毎週」が期待より多く(70 人 > 期待 56 人)、
> 「未経験」が期待より少ない(20 人 < 期待 28 人)ことから、**上級生ほど利用が定着している**傾向が読み取れる。

**注意**: カイ二乗検定の正規近似が使えるのは、**期待度数がおおむね 5 以上**のときです。
小さいセルがある場合は、カテゴリを併合するか、2×2 なら Fisher の正確検定(`stats.fisher_exact`)を使います。
なお、カイ二乗検定は片側・両側を選ぶのではなく、**ずれの大きさを常に右側の裾**で評価します。

## まとめ — 12 の問題の比較と検定の選び方

| 問題 | 検定 | t 値(自由度) | p 値 | 結論 |
|---|---|---|---|---|
| 例題: パソコン購入価格 vs 14 万円 | 1 標本 t・片側 | 1.863 (179) | 0.032 | 棄却(高い) |
| 練習 1: 学食支払額 vs 1,500 円 | 1 標本 t・片側 | 2.00 (35) | 0.027 | 棄却(高い) |
| 練習 2: 自習時間 1 年生 vs 3 年生 | Welch・両側 | −2.65 (≈145) | 0.009 | 棄却(差あり) |
| 練習 3: 動画配信料金 A 大学 vs B 大学 | Welch・両側 | 2.83 (≈81) | 0.006 | 棄却(差あり) |
| 練習 4: 電子書籍 20〜39 歳 vs 40〜59 歳 | Welch・両側 | −2.65 (≈77) | 0.010 | 棄却(差あり) |
| 練習 5: 生成 AI サービス A vs B(同じ学生) | 対応のある t・両側 | 4.79 (149) | < 0.001 | 棄却(A が高い) |
| 練習 6: デザイン A vs B の評価のばらつき | 分散の F・両側 | F = 2.00 (49, 49) | 0.017 | 棄却(A が割れる) |
| 練習 7: 広告デザイン × 割引率 | 二元配置分散分析 | A: 77.9 / B: 185.3 / A×B: 1.07 | <0.001 / <0.001 / 0.39 | 主効果は有意、交互作用なし |
| 練習 8: 広告費と新規顧客獲得数 | 単回帰・傾きの t | 49.9 (14) | < 0.001 | 棄却(傾き 6.2、R² = 0.994) |
| 練習 9: 毎週利用する学生の割合 vs 30% | 母比率の z・両側 | z = 3.06 | 0.002 | 棄却(全国より高い) |
| 練習 10: LP の A/B テスト | 2 比率の差の z・両側 | z = 2.31 | 0.021 | 棄却(A 案が高い) |
| 練習 11: 学年 × 利用頻度の関連 | カイ二乗独立性 | χ² = 12.05 (2) | 0.002 | 棄却(関連あり) |

**検定の選び方の要点**:

- 「〜より高い / 低い」→ **片側**、「違いがあるか / 差があるか」→ **両側**
- 母標準偏差が不明なら z 検定ではなく **t 検定**
- **同じ対象を 2 回測った**データ → 差を取って**対応のある t 検定**
- **別々のグループ**の比較で母分散が等しいと言えないなら **Welch 流**(`equal_var=False`)。
  実務では最初から Welch を使うのが安全、というのが現在の標準的な考え方です
- 比べたいのが平均ではなく**ばらつき** → 分散比の **F 検定**(正規性が前提。生データがあれば Levene 検定も)
- **3 グループ以上**や **2 つの要因**の平均の比較 → **分散分析(ANOVA)**。交互作用まで調べられる
- **数値と数値の関係**を式にしたい → **回帰分析**(傾きの t 検定で関係の有意性を確認。相関は因果を意味しない)
- 「はい / いいえ」の**割合**の検定 → **比率の z 検定**(小標本なら二項検定)。A/B テストは 2 比率の差
- **カテゴリ × カテゴリ**の関連 → 分割表の**カイ二乗独立性検定**(期待度数 5 以上が目安)

対応のない 3 群以上の比率、ノンパラメトリック検定、多重比較などのさらに進んだ話題は
`statistics/statistics-python.ipynb` で幅広く練習できます。